# 02C - Asset / Equipment Finance LGD

This notebook adds a dedicated asset/equipment-finance LGD segment under the parent commercial cash-flow framework.

Asset categories covered:
- vehicles
- standard equipment
- specialised machinery


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_commercial_data
from src.commercial_data_controls import assign_framework_segment, run_commercial_data_controls
from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(72)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 160)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
TABLE_DIR = os.path.join(OUTPUT_DIR, 'tables')
FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")


## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy:** observed workout inputs -> approved proxies -> conservative fallback with disclosure.
- **Proxy logic:** asset type, condition, secondary liquidity, and remarketing assumptions are synthetic portfolio proxies.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status:** portfolio-project baseline only; not calibrated to internal repossession and resale history.


## Objective
Build an interview-ready asset/equipment finance LGD segment where recoveries are tied to repossession and remarketing of financed assets.

## Drivers
- asset type and age
- residual / balloon exposure
- repossession and enforcement cost
- remarketing discount and secondary-market liquidity
- condition/location and borrower operational dependence

## Logic
- EAD reflects instalment-style drawn balances plus modest residual/headroom crystallisation.
- LGD is based on discounted sale proceeds net of repossession/holding costs and legal processing costs.

## Downturn
- Stress assumes weaker resale liquidity, higher remarketing haircuts, and longer repossession-to-sale timing.

## Validation
- Range checks, balance checks, and timing sanity checks are exported for reviewer governance.

## Limitations
- Portfolio-project proxy assumptions are used; production calibration requires internal repossession, resale, and deficiency data.


## Why Asset Finance Differs from Generic Secured Commercial Lending

- Recovery is tied to **specific financed assets** rather than broad enterprise collateral packages.
- Asset age, condition, and secondary-market depth directly affect resale proceeds.
- Residual/balloon exposures can increase loss severity if remarketing outcomes are weak.
- Workout path is typically repossession-and-sale focused, often shorter but more collateral-quality sensitive.


In [ ]:
# Base commercial portfolio and asset/equipment segment selection
all_loans, all_cashflows = generate_commercial_data(n_loans=420, seed=43)
all_loans = all_loans.copy()

all_loans['icr'] = pd.to_numeric(all_loans['interest_coverage_ratio'], errors='coerce')
all_loans['industry_risk_bucket'] = all_loans['industry_risk_level'].fillna('Medium')
all_loans['watchlist_flag'] = (
    all_loans['default_trigger'].isin(['Covenant Breach', 'Voluntary Administration', 'Receivership'])
    | (all_loans['icr'] < 1.35)
).astype(int)
all_loans['framework_segment'] = assign_framework_segment(all_loans)

asset = all_loans[all_loans['framework_segment'] == 'Asset / Equipment Finance'].copy()

if len(asset) == 0:
    raise ValueError('No asset/equipment segment rows found for current synthetic sample.')

print('All loans:', len(all_loans))
print('Asset/equipment segment loans:', len(asset))
display(asset[['loan_id', 'facility_type', 'security_type', 'ead', 'drawn_balance', 'undrawn_amount']].head(10))


In [ ]:
# Asset drivers and EAD logic
rng = np.random.default_rng(72)

asset['asset_type'] = rng.choice(
    ['Vehicles', 'Standard Equipment', 'Specialised Machinery'],
    size=len(asset),
    p=[0.45, 0.35, 0.20],
)

type_age = {
    'Vehicles': (1.0, 7.0),
    'Standard Equipment': (2.0, 10.0),
    'Specialised Machinery': (3.0, 14.0),
}
asset['asset_age_years'] = [rng.uniform(*type_age[t]) for t in asset['asset_type']]

asset['residual_balloon_pct'] = (
    0.08
    + 0.10 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    + 0.05 * asset['asset_type'].eq('Vehicles').astype(int)
    + rng.normal(0.0, 0.03, len(asset))
).clip(0.00, 0.35)

asset['repossession_cost_pct'] = (
    0.020
    + 0.018 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    + 0.012 * asset['watchlist_flag']
    + rng.normal(0.0, 0.005, len(asset))
).clip(0.010, 0.090)

asset['remarketing_discount_pct'] = (
    0.120
    + 0.090 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    + 0.050 * (asset['asset_age_years'] / 12.0).clip(0.0, 1.0)
    + 0.040 * asset['watchlist_flag']
    + rng.normal(0.0, 0.020, len(asset))
).clip(0.05, 0.50)

asset['secondary_market_liquidity'] = (
    0.78
    - 0.32 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    - 0.10 * (asset['asset_age_years'] / 12.0).clip(0.0, 1.0)
    + rng.normal(0.0, 0.06, len(asset))
).clip(0.10, 0.98)

asset['asset_condition_location_score'] = (
    0.70
    - 0.10 * (asset['asset_age_years'] / 12.0).clip(0.0, 1.0)
    - 0.08 * asset['watchlist_flag']
    + rng.normal(0.0, 0.06, len(asset))
).clip(0.10, 0.98)

asset['operational_dependence_score'] = (
    0.45
    + 0.22 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    + 0.08 * asset['industry_risk_bucket'].map({'Low': 0.0, 'Medium': 0.5, 'Elevated': 1.0}).fillna(0.5)
    + rng.normal(0.0, 0.06, len(asset))
).clip(0.05, 0.98)

asset['liquidity_band'] = pd.cut(
    asset['secondary_market_liquidity'],
    bins=[0.0, 0.35, 0.60, 0.80, 1.00],
    labels=['Very Low', 'Low', 'Medium', 'High'],
    include_lowest=True,
)

asset['ead_base'] = (
    asset['drawn_balance']
    + 0.20 * asset['ccf'] * asset['undrawn_amount']
    + 0.18 * asset['residual_balloon_pct'] * asset['facility_limit']
)
asset['ead_base'] = np.maximum(asset['ead_base'], asset['drawn_balance'])
asset['ead_base'] = np.minimum(asset['ead_base'], asset['facility_limit'] * 1.05)

asset['ead_downturn'] = (
    asset['ead_base']
    * (1.02 + 0.04 * asset['watchlist_flag'] + 0.03 * asset['operational_dependence_score'])
)
asset['ead_downturn'] = np.maximum(asset['ead_downturn'], asset['drawn_balance'])
asset['ead_downturn'] = np.minimum(asset['ead_downturn'], asset['facility_limit'] * 1.10)

asset['ead_uplift_pct'] = ((asset['ead_downturn'] - asset['ead_base']) / asset['ead_base'].replace(0, np.nan)).fillna(0.0).clip(0.0, 1.5)

ead_summary = (
    asset.groupby('asset_type', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_drawn': g['drawn_balance'].sum(),
                'total_ead_base': g['ead_base'].sum(),
                'total_ead_downturn': g['ead_downturn'].sum(),
                'ead_weighted_uplift_pct': exposure_weighted_average(g, 'ead_uplift_pct', 'ead_base'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values('total_ead_base', ascending=False)
)

print('=== Asset/Eq EAD Summary ===')
display(ead_summary.round(4))


In [ ]:
# Recovery and LGD logic (base/downturn/final)
specialised_flag = asset['asset_type'].eq('Specialised Machinery').astype(int)
vehicle_flag = asset['asset_type'].eq('Vehicles').astype(int)

asset_age_scaled = (asset['asset_age_years'] / 12.0).clip(0.0, 1.2)

asset['repossession_months'] = (
    2.5
    + 4.0 * (1.0 - asset['secondary_market_liquidity'])
    + 2.0 * specialised_flag
    + 1.5 * asset['watchlist_flag']
    + 1.2 * asset_age_scaled
    + np.random.default_rng(721).normal(0.0, 0.8, len(asset))
).clip(1.0, 24.0)

asset['sale_timing_months'] = (
    asset['repossession_months']
    + 1.5
    + 3.0 * (1.0 - asset['secondary_market_liquidity'])
    + 1.5 * specialised_flag
).clip(2.0, 36.0)

asset['asset_haircut_proxy'] = (
    asset['remarketing_discount_pct']
    + 0.10 * specialised_flag
    + 0.06 * asset_age_scaled
    + 0.05 * (1.0 - asset['asset_condition_location_score'])
).clip(0.08, 0.70)

asset['sale_proceeds_pct_of_ead'] = (
    (1.0 - asset['asset_haircut_proxy'])
    * (0.62 + 0.30 * asset['secondary_market_liquidity'])
    * (0.65 + 0.30 * asset['asset_condition_location_score'])
).clip(0.05, 0.98)

asset['discount_factor_proxy'] = (1.0 + asset['discount_rate']) ** (asset['sale_timing_months'] / 12.0)
asset['discounted_recovery_pct'] = (asset['sale_proceeds_pct_of_ead'] / asset['discount_factor_proxy']).clip(0.03, 0.98)

asset['enforcement_holding_cost_pct'] = (
    asset['repossession_cost_pct']
    + 0.012
    + 0.010 * (asset['sale_timing_months'] / 24.0).clip(0.0, 1.5)
    + 0.012 * specialised_flag
).clip(0.015, 0.18)

asset['liquidation_loss_proxy'] = (
    1.0
    - asset['discounted_recovery_pct']
    + asset['enforcement_holding_cost_pct']
    + 0.05 * asset['residual_balloon_pct']
    + 0.04 * asset['operational_dependence_score']
).clip(0.05, 0.99)

asset['lgd_base_economic'] = (
    0.45 * asset['realised_lgd']
    + 0.55 * asset['liquidation_loss_proxy']
).clip(0.05, 0.95)

asset['sale_timing_months_downturn'] = (
    asset['sale_timing_months']
    * (1.10 + 0.08 * specialised_flag + 0.04 * (1.0 - asset['secondary_market_liquidity']))
).clip(2.0, 48.0)

asset['downturn_scalar'] = (
    1.00
    + 0.16 * asset['asset_haircut_proxy']
    + 0.08 * ((asset['sale_timing_months_downturn'] - asset['sale_timing_months']) / 24.0).clip(0.0, 1.0)
    + 0.06 * asset['operational_dependence_score']
    + 0.03 * vehicle_flag
).clip(1.00, 1.55)

asset['lgd_downturn'] = (asset['lgd_base_economic'] * asset['downturn_scalar']).clip(0.0, 1.0)

asset['moc_addon'] = 0.020 + 0.010 * asset['watchlist_flag']
asset['supervisory_floor_proxy'] = asset['asset_type'].map({
    'Vehicles': 0.25,
    'Standard Equipment': 0.35,
    'Specialised Machinery': 0.45,
}).fillna(0.35)
asset['lgd_final_regulatory'] = np.maximum(asset['lgd_downturn'] + asset['moc_addon'], asset['supervisory_floor_proxy']).clip(0.0, 1.0)

def weighted_lgd(df, group_col):
    rows = []
    for k, g in df.groupby(group_col, observed=True):
        rows.append({
            'dimension': group_col,
            'bucket': str(k),
            'loan_count': len(g),
            'total_ead_base': g['ead_base'].sum(),
            'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base_economic', 'ead_base'),
            'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead_base'),
            'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
        })
    return pd.DataFrame(rows)

weighted_by_asset = weighted_lgd(asset, 'asset_type')
weighted_by_liquidity = weighted_lgd(asset, 'liquidity_band')
segment_summary = pd.concat([weighted_by_asset, weighted_by_liquidity], ignore_index=True)

base_vs_downturn = pd.DataFrame([
    {
        'ead_weighted_lgd_base': exposure_weighted_average(asset, 'lgd_base_economic', 'ead_base'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(asset, 'lgd_downturn', 'ead_base'),
        'ead_weighted_lgd_final': exposure_weighted_average(asset, 'lgd_final_regulatory', 'ead_base'),
    }
])
base_vs_downturn['downturn_sensitivity_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_downturn'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)
base_vs_downturn['final_minus_base_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_final'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)

print('=== Asset/Eq Weighted LGD by Asset Type ===')
display(weighted_by_asset.round(4))
print('=== Asset/Eq Weighted LGD by Liquidity Band ===')
display(weighted_by_liquidity.round(4))
print('=== Base vs Downturn ===')
display(base_vs_downturn.round(4))


In [ ]:
# Sensitivity analysis: remarketing discount and liquidity
def run_asset_sensitivity(df, discount_add=0.0, liquidity_shift=0.0):
    x = df.copy()
    remarketing = (x['remarketing_discount_pct'] + discount_add).clip(0.01, 0.80)
    liquidity = (x['secondary_market_liquidity'] + liquidity_shift).clip(0.05, 1.00)

    haircut = (
        remarketing
        + 0.10 * x['asset_type'].eq('Specialised Machinery').astype(int)
        + 0.06 * (x['asset_age_years'] / 12.0).clip(0.0, 1.2)
        + 0.05 * (1.0 - x['asset_condition_location_score'])
    ).clip(0.08, 0.80)

    proceeds = (
        (1.0 - haircut)
        * (0.62 + 0.30 * liquidity)
        * (0.65 + 0.30 * x['asset_condition_location_score'])
    ).clip(0.03, 0.98)

    rec_months = (x['sale_timing_months'] * (1.00 + 0.25 * discount_add - 0.20 * liquidity_shift)).clip(2.0, 52.0)
    disc = (1.0 + x['discount_rate']) ** (rec_months / 12.0)

    lgd_base_s = (
        0.45 * x['realised_lgd']
        + 0.55 * (
            1.0
            - (proceeds / disc)
            + x['enforcement_holding_cost_pct']
            + 0.05 * x['residual_balloon_pct']
            + 0.04 * x['operational_dependence_score']
        )
    ).clip(0.05, 0.97)

    lgd_down_s = (
        lgd_base_s
        * (1.00 + 0.16 * haircut + 0.05 * x['operational_dependence_score'])
    ).clip(0.0, 1.0)

    lgd_final_s = np.maximum(
        lgd_down_s + 0.020 + 0.010 * x['watchlist_flag'],
        x['supervisory_floor_proxy'],
    ).clip(0.0, 1.0)

    y = x.assign(lgd_base_s=lgd_base_s, lgd_down_s=lgd_down_s, lgd_final_s=lgd_final_s)
    return {
        'ead_weighted_lgd_base': exposure_weighted_average(y, 'lgd_base_s', 'ead_base'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(y, 'lgd_down_s', 'ead_base'),
        'ead_weighted_lgd_final': exposure_weighted_average(y, 'lgd_final_s', 'ead_base'),
    }

scenarios = [
    {'scenario': 'base', 'discount_add': 0.00, 'liquidity_shift': 0.00},
    {'scenario': 'remarketing_discount_+10pp', 'discount_add': 0.10, 'liquidity_shift': 0.00},
    {'scenario': 'liquidity_-10pp', 'discount_add': 0.00, 'liquidity_shift': -0.10},
    {'scenario': 'combined_stress', 'discount_add': 0.10, 'liquidity_shift': -0.10},
]

sensitivity_rows = []
for s in scenarios:
    m = run_asset_sensitivity(asset, discount_add=s['discount_add'], liquidity_shift=s['liquidity_shift'])
    sensitivity_rows.append({'scenario': s['scenario'], **m})

sensitivity = pd.DataFrame(sensitivity_rows)
print('=== Asset/Eq Sensitivity ===')
display(sensitivity.round(4))


In [ ]:
# Monitoring, validation, charts, exports
asset['default_year'] = pd.to_datetime(asset['default_date']).dt.year
monitoring = (
    asset.groupby('default_year', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead_base': g['ead_base'].sum(),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
                'mean_sale_timing_months': g['sale_timing_months'].mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

core_controls = run_commercial_data_controls(
    asset,
    None,
    segment_col='framework_segment',
    extra_probability_cols=['asset_condition_location_score', 'operational_dependence_score'],
    extra_haircut_cols=['remarketing_discount_pct', 'asset_haircut_proxy', 'residual_balloon_pct'],
)

extra_checks = pd.DataFrame([
    {'check_name': 'ead_base_not_below_drawn', 'passed': (asset['ead_base'] >= asset['drawn_balance']).all(), 'failed_rows': int((asset['ead_base'] < asset['drawn_balance']).sum()), 'detail': 'ead_base >= drawn_balance'},
    {'check_name': 'ead_downturn_not_below_drawn', 'passed': (asset['ead_downturn'] >= asset['drawn_balance']).all(), 'failed_rows': int((asset['ead_downturn'] < asset['drawn_balance']).sum()), 'detail': 'ead_downturn >= drawn_balance'},
    {'check_name': 'downturn_sale_timing_not_shorter', 'passed': (asset['sale_timing_months_downturn'] >= asset['sale_timing_months']).all(), 'failed_rows': int((asset['sale_timing_months_downturn'] < asset['sale_timing_months']).sum()), 'detail': 'downturn sale timing >= base sale timing'},
    {'check_name': 'lgd_downturn_not_below_base', 'passed': (asset['lgd_downturn'] >= asset['lgd_base_economic']).all(), 'failed_rows': int((asset['lgd_downturn'] < asset['lgd_base_economic']).sum()), 'detail': 'downturn LGD >= base LGD'},
])

validation_checks = pd.concat([core_controls, extra_checks], ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(weighted_by_asset['bucket'], weighted_by_asset['ead_weighted_lgd_final'], color='#4c72b0')
ax.set_title('Asset/Equipment LGD: Weighted Final LGD by Asset Type')
ax.set_xlabel('Asset Type')
ax.set_ylabel('EAD-weighted Final LGD')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'asset_equipment_weighted_lgd_by_asset_type.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(ead_summary['asset_type'], ead_summary['ead_weighted_uplift_pct'], color='#55a868')
ax.set_title('Asset/Equipment EAD Uplift by Asset Type')
ax.set_xlabel('Asset Type')
ax.set_ylabel('EAD Uplift %')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'asset_equipment_ead_uplift_by_asset_type.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

loan_level_output = asset[[
    'loan_id', 'facility_type', 'security_type', 'framework_segment', 'industry', 'industry_risk_bucket',
    'asset_type', 'asset_age_years', 'residual_balloon_pct', 'repossession_cost_pct',
    'remarketing_discount_pct', 'secondary_market_liquidity', 'asset_condition_location_score',
    'operational_dependence_score', 'liquidity_band', 'drawn_balance', 'undrawn_amount',
    'ead_base', 'ead_downturn', 'ead_uplift_pct', 'repossession_months', 'sale_timing_months',
    'sale_timing_months_downturn', 'asset_haircut_proxy', 'discounted_recovery_pct',
    'enforcement_holding_cost_pct', 'realised_lgd', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory',
]].copy()

loan_level_output.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_segment_summary.csv'), index=False)
ead_summary.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_ead_summary.csv'), index=False)
base_vs_downturn.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_base_vs_downturn.csv'), index=False)
sensitivity.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_sensitivity.csv'), index=False)
monitoring.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_monitoring_by_year.csv'), index=False)
validation_checks.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_validation_checks.csv'), index=False)

print('=== Validation Checks ===')
display(validation_checks)

print('Saved asset/equipment outputs:')
print('- asset_equipment_finance_loan_level_output.csv')
print('- asset_equipment_finance_segment_summary.csv')
print('- asset_equipment_finance_ead_summary.csv')
print('- asset_equipment_finance_base_vs_downturn.csv')
print('- asset_equipment_finance_sensitivity.csv')
print('- asset_equipment_finance_monitoring_by_year.csv')
print('- asset_equipment_finance_validation_checks.csv')
